# Sujet 07 — Détecter la transaction d'un client

**Question :** peut-on repérer, dans les 200 variables anonymisées d'un client, ceux qui vont réaliser une
transaction, et de combien une méthode soignée bat-elle la règle sans modèle (plancher AUC ≈ 0,5639) ?

Le notebook suit les trois blocs de l'énoncé :

| Bloc | Rôle dominant | Contenu |
|---|---|---|
| 0 | constructeur | configuration, chargement, contrôles |
| 1 | mesureur | métrique, coûts, plancher, protocole de validation |
| 2 | constructeur + interprète | AUC de chaque variable, courbe 1 → 200 variables, ≥ 8 essais, analyse d'erreurs |
| 3 | interprète | trois mutations nommées, robustesse, décision finale sur les 40 000 clients |

Tout le code réutilisable est dans `santander.py` ; `pipeline.py` refait la décision finale seul,
en moins de 10 minutes sur CPU.

## 0 · Configuration — *la seule cellule à changer pour passer d'une version à l'autre*

In [ ]:
from pathlib import Path
import time, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import santander as S

DOSSIER   = Path("donnees")
TRAIN     = DOSSIER / "train.csv" if (DOSSIER / "train.csv").exists() else DOSSIER / "train_extrait.csv"
A_DECIDER = DOSSIER / "a_decider.csv"
SORTIES   = Path("sorties"); SORTIES.mkdir(exist_ok=True)

GRAINE        = S.GRAINE
PART_TEST     = 0.20      # test interne, mis de côté et touché UNE fois, à la fin
N_PLIS        = 5         # validation croisée stratifiée sur le reste
COUT_FP       = 2.0       # € : une sollicitation inutile (contact, remise, lassitude du client)
COUT_FN       = 20.0      # € : une transaction manquée (marge perdue)
N_BINS        = 64        # finesse du modèle retenu
K_VARIABLES   = None      # None = décidé par la courbe du bloc 2

print("fichier d'apprentissage :", TRAIN)

In [ ]:
t0 = time.perf_counter()
train = S.charger(TRAIN)
decider = S.charger(A_DECIDER)
VARS = S.colonnes_var(train)
X = train[VARS].to_numpy(np.float32)
y = train[S.CIBLE].to_numpy(int)
print(f"train : {train.shape}   a_decider : {decider.shape}   chargé en {time.perf_counter()-t0:.1f} s")
print("valeurs manquantes :", int(train.isna().sum().sum()), "/", int(decider.isna().sum().sum()))
print(f"taux de transaction : {y.mean():.4f}  ({y.sum()} sur {len(y)})")
print(f"plage des variables : {X.min():.1f} à {X.max():.1f} ; variables avec des négatifs : {(X.min(0) < 0).sum()}")
assert list(decider.columns) == [c for c in train.columns if c != S.CIBLE], "colonnes différentes"
assert not set(train[S.ID]) & set(decider[S.ID]), "un client est dans les deux fichiers"

## 1 · La décision et la mesure *(mesureur)*

### 1.1 La métrique : AUC
L'AUC mesure la probabilité qu'un client **qui fera** la transaction soit classé **au-dessus** d'un client qui
ne la fera pas. Elle récompense **l'ordre** ; elle ignore le seuil, l'étalonnage des probabilités et les coûts.

**Pourquoi pas l'exactitude ?** Avec 10 % de positifs, répondre « non » à tout le monde donne 90 %
d'exactitude et ne repère personne. L'exactitude récompense ce classement vide ; l'AUC le note 0,5.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
print("exactitude du « personne ne fera de transaction » :", round(accuracy_score(y, np.zeros_like(y)), 4))
print("AUC du même classement                           :", roc_auc_score(y, np.zeros(len(y))))

### 1.2 Le coût
Hypothèses (à défendre, modifiables dans la cellule 0) :
- **sollicitation inutile (faux positif)** : **2 €** — coût du contact et usure du client ;
- **transaction manquée (faux négatif)** : **20 €** — marge moyenne perdue.

Une vente manquée coûte donc **10 fois** une sollicitation inutile : il vaut la peine de solliciter un client dès
que sa probabilité dépasse 2 / (2 + 20) ≈ **9 %** si les probabilités étaient bien étalonnées. Le seuil sera
choisi sur les scores hors-pli, en coût total, pas en probabilité supposée.

### 1.3 Le protocole de validation
1. **Test interne** : 20 % de `train.csv`, tiré **stratifié** (même taux de 10 %), mis de côté **avant tout** et
   utilisé une seule fois, au bloc 3.
2. **Validation croisée stratifiée à 5 plis** sur les 80 % restants pour tous les essais : on rapporte la moyenne et
   l'écart-type. L'écart-type dit si une différence entre deux essais est réelle.
3. Tout ce qui apprend de la cible (choix de variable, tranches, modèle, sélection) est appris **dans le pli
   d'entraînement** : c'est la règle « pas de fuite ».

Pas de découpage temporel : la table n'a ni date ni ordre connu, les clients sont indépendants.

In [ ]:
from sklearn.model_selection import train_test_split
idx = np.arange(len(y))
i_dev, i_test = train_test_split(idx, test_size=PART_TEST, stratify=y, random_state=GRAINE)
Xd, yd, Xt, yt = X[i_dev], y[i_dev], X[i_test], y[i_test]
print(f"développement : {len(yd)} clients ({yd.mean():.4f})   test interne : {len(yt)} clients ({yt.mean():.4f})")
journal = S.Journal()

### 1.4 Le plancher : la règle sans modèle
Sans modèle, la banque ne peut que **trier ses clients sur une variable** : on prend celle qui sépare le mieux
(mesurée sur le pli d'entraînement), dans le bon sens. C'est le plancher à battre.

In [ ]:
auc, sd, sec = S.score_cv(S.RegleUneVariable, Xd, yd, N_PLIS)
regle = S.RegleUneVariable().fit(Xd, yd)
print(f"plancher : AUC {auc:.4f} ± {sd:.4f}   variable choisie : {VARS[regle.j_]}  (sens {'+' if regle.signe_>0 else '−'})")
journal.noter("E0 plancher : trier sur une variable", auc, sd, secondes=sec, verdict="référence")
PLANCHER = auc

## 2 · Le pipeline et les essais *(constructeur, puis interprète)*

### 2.1 Que vaut chaque variable prise seule ?
AUC de chaque variable brute (mesurée sur la partie développement).

In [ ]:
auc_var = pd.Series([roc_auc_score(yd, Xd[:, j]) for j in range(Xd.shape[1])], index=VARS)
force = (auc_var - 0.5).abs().sort_values(ascending=False)
print("les 10 plus fortes :"); print(force.head(10).round(4).to_string())
print(f"\nmédiane : {force.median():.4f}   variables à moins de 0,51 : {(force < 0.01).sum()}")
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(range(len(force)), 0.5 + force.values, width=1.0, color="#3a6ea5")
ax.axhline(PLANCHER, color="#c0392b", lw=1, ls="--", label=f"plancher {PLANCHER:.4f}")
ax.set_ylim(0.49, max(0.6, 0.5 + force.max() + 0.01)); ax.set_xlabel("variables, de la plus forte à la plus faible")
ax.set_ylabel("AUC seule"); ax.legend(); ax.set_title("Chaque variable seule est à peine meilleure que le hasard")
plt.tight_layout(); plt.savefig(SORTIES / "auc_par_variable.png", dpi=120); plt.show()

**Les variables sont-elles liées entre elles ?** Si elles le sont peu, chacune apporte une information
*nouvelle*, et les additionner doit payer — c'est l'hypothèse d'un Bayes naïf.

In [ ]:
corr = np.corrcoef(Xd[:20000].T)
hors_diag = np.abs(corr[np.triu_indices_from(corr, 1)])
print(f"corrélation absolue entre variables : max {hors_diag.max():.3f}, médiane {np.median(hors_diag):.4f}")

**Le lien avec la cible est-il monotone ?** Une régression logistique suppose que plus la variable monte, plus
(ou moins) la transaction est probable. On compare, pour les plus fortes variables, le taux de transaction par
tranche de valeurs.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, v in zip(axes, force.index[:4]):
    q = pd.qcut(pd.Series(Xd[:, VARS.index(v)]), 30, duplicates="drop")
    taux = pd.Series(yd).groupby(q.cat.codes).mean()
    ax.plot(taux.index, taux.values, marker=".", color="#3a6ea5"); ax.axhline(yd.mean(), color="grey", lw=.8)
    ax.set_title(v); ax.set_xlabel("tranche (30 quantiles)")
axes[0].set_ylabel("taux de transaction"); plt.tight_layout(); plt.savefig(SORTIES / "forme_des_liens.png", dpi=120); plt.show()

### 2.2 Les essais
Chaque essai est mesuré par la même validation croisée et laisse une ligne dans le journal, **qu'il réussisse
ou non**.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import HistGradientBoostingClassifier

ESSAIS = {
    "E1 régression logistique (variables standardisées)":
        (lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=500)), "linéaire, lien supposé monotone"),
    "E2 Bayes naïf gaussien (variables brutes)":
        (lambda: GaussianNB(), "indépendance, densités supposées gaussiennes"),
    "E3 Bayes naïf gaussien après transformation en quantiles normaux":
        (lambda: make_pipeline(QuantileTransformer(output_distribution="normal", subsample=50000, random_state=GRAINE), GaussianNB()), "corrige la forme des marges"),
    "E4 gradient boosting d'histogrammes (sklearn, 300 arbres)":
        (lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=15, random_state=GRAINE), "interactions possibles, coûteux"),
    "E5 Bayes naïf par quantiles, 64 tranches, sans lissage entre voisines":
        (lambda: S.BayesNaifQuantiles(n_bins=64, voisinage=0), "chaque tranche jugée seule : bruit"),
    "E6 Bayes naïf par quantiles, 64 tranches lissées":
        (lambda: S.BayesNaifQuantiles(n_bins=64, lissage=50, voisinage=2), "notre modèle"),
    "E7 Bayes naïf par quantiles, 256 tranches, peu lissé":
        (lambda: S.BayesNaifQuantiles(n_bins=256, lissage=2, voisinage=0), "tranches fines : sur-apprentissage"),
}
for nom, (fab, note) in ESSAIS.items():
    auc, sd, sec = S.score_cv(fab, Xd, yd, N_PLIS)
    journal.noter(nom, auc, sd, secondes=sec, note=note)
    print(f"{auc:.4f} ± {sd:.4f}   {sec:6.1f} s   {nom}")

### 2.3 La courbe : combien de variables garder ?
Pour k = 1, 2, 5, 10, 20, 50, 100, 150, 200 : on garde les k variables les plus fortes **choisies dans le pli
d'entraînement**, puis on mesure le modèle retenu. C'est cette courbe qui décide de la sélection.

In [ ]:
KS = [1, 2, 5, 10, 20, 50, 100, 150, 200]
courbe = []
for k in KS:
    fab = lambda k=k: make_pipeline(S.SelectionAUC(k), S.BayesNaifQuantiles(n_bins=N_BINS))
    auc, sd, sec = S.score_cv(fab, Xd, yd, N_PLIS)
    courbe.append((k, auc, sd, sec)); print(f"k={k:3d}  AUC {auc:.4f} ± {sd:.4f}  {sec:5.1f} s")
courbe = pd.DataFrame(courbe, columns=["k", "auc", "ecart", "secondes"])
courbe.to_csv(SORTIES / "courbe_nb_variables.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.errorbar(courbe.k, courbe.auc, yerr=courbe.ecart, marker="o", color="#3a6ea5", capsize=3)
ax.axhline(PLANCHER, color="#c0392b", ls="--", lw=1, label="plancher")
ax.set_xscale("log"); ax.set_xticks(KS); ax.set_xticklabels(KS); ax.set_xlabel("nombre de variables gardées")
ax.set_ylabel("AUC (5 plis)"); ax.set_title("Le gain ne sature pas : chaque variable apporte un peu"); ax.legend()
plt.tight_layout(); plt.savefig(SORTIES / "courbe_nb_variables.png", dpi=120); plt.show()

**Règle de choix :** on garde le plus petit k dont l'AUC est à moins d'un écart-type de la meilleure. Garder
moins de variables doit se payer en AUC ; si ça ne coûte rien, on garde moins.

In [ ]:
meilleur = courbe.loc[courbe.auc.idxmax()]
if K_VARIABLES is None:
    K_VARIABLES = int(courbe[courbe.auc >= meilleur.auc - meilleur.ecart].k.min())
print(f"meilleur k = {int(meilleur.k)} (AUC {meilleur.auc:.4f}) → k retenu = {K_VARIABLES}")
row = courbe[courbe.k == K_VARIABLES].iloc[0]
journal.noter(f"E8 Bayes naïf par quantiles, {K_VARIABLES} variables choisies dans le pli", row.auc, row.ecart,
              secondes=row.secondes, note="sélection décidée par la courbe")
row = courbe[courbe.k == 20].iloc[0]
journal.noter("E9 Bayes naïf par quantiles, 20 meilleures variables seulement", row.auc, row.ecart,
              secondes=row.secondes, note="« les variables fortes suffisent » : faux")

### 2.4 Deux idées de plus, mesurées
- **E10** : apprendre un **poids par variable** (régression logistique sur les 200 log-rapports, appris dans le
  pli) au lieu de les additionner tels quels.
- **E11** : ajouter des interactions par un boosting **sur les log-rapports** du Bayes naïf.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
class Empile(BaseEstimator, ClassifierMixin):
    """Bayes naïf par quantiles → contributions → second modèle. Tout est appris sur le pli d'entraînement :
    les contributions d'entraînement sont calculées hors-pli (3 plis internes) pour ne pas se recracher."""
    def __init__(self, second): self.second = second
    def fit(self, X, y):
        from sklearn.model_selection import StratifiedKFold
        from sklearn.base import clone
        Z = np.zeros_like(X, dtype=np.float32)
        for tr, va in StratifiedKFold(3, shuffle=True, random_state=GRAINE).split(X, y):
            Z[va] = S.BayesNaifQuantiles(N_BINS).fit(X[tr], y[tr]).contributions(X[va])
        self.nb_ = S.BayesNaifQuantiles(N_BINS).fit(X, y)
        self.m_ = clone(self.second).fit(Z, y); self.classes_ = np.array([0, 1]); return self
    def predict_proba(self, X): return self.m_.predict_proba(self.nb_.contributions(X))

for nom, second, note in [
    ("E10 poids par variable (logistique sur les log-rapports)", LogisticRegression(C=1.0, max_iter=1000), "corrige les redondances éventuelles"),
    ("E11 boosting sur les log-rapports", HistGradientBoostingClassifier(max_iter=200, learning_rate=0.06, max_leaf_nodes=8, random_state=GRAINE), "cherche des interactions"),
]:
    auc, sd, sec = S.score_cv(lambda s=second: Empile(s), Xd, yd, N_PLIS)
    journal.noter(nom, auc, sd, secondes=sec, note=note); print(f"{auc:.4f} ± {sd:.4f}   {sec:6.1f} s   {nom}")

### 2.5 Le tableau des essais

In [ ]:
tab = journal.tableau()
ref = tab.auc_cv.max()
tab["vs_plancher"] = (tab.auc_cv - PLANCHER).round(4)
tab["verdict"] = np.where(tab.essai.str.startswith("E0"), "référence",
                 np.where(tab.auc_cv >= ref - tab.ecart_type, "retenu possible",
                 np.where(tab.auc_cv > PLANCHER + 0.05, "moins bon", "échec")))
tab.to_csv(SORTIES / "essais.csv", index=False)
pd.set_option("display.max_colwidth", 70); tab

In [ ]:
# Le modèle retenu : le meilleur compromis AUC / simplicité du tableau (modifiable ici, en une ligne).
FABRIQUE = lambda: make_pipeline(S.SelectionAUC(K_VARIABLES), S.BayesNaifQuantiles(n_bins=N_BINS))
NOM_RETENU = f"Bayes naïf par quantiles, {N_BINS} tranches, {K_VARIABLES} variables"
auc, sd, sec, oof = S.score_cv(FABRIQUE, Xd, yd, N_PLIS, retour_oof=True)
print(f"retenu : {NOM_RETENU}  —  AUC hors-pli {auc:.4f} ± {sd:.4f}  ({sec:.1f} s pour 5 plis)")

### 2.6 Du score à la décision : le seuil
On choisit **quelle part des clients solliciter** en minimisant le coût total sur les scores hors-pli.
On décide en **part de clients** (les x % les mieux classés) plutôt qu'en probabilité : c'est stable quand le modèle
est réentraîné sur plus de données.

In [ ]:
seuil, cout, n_sel = S.seuil_cout_minimal(yd, oof, COUT_FP, COUT_FN)
PART_SOLLICITEE = n_sel / len(yd)
cout_tous = (1 - yd).sum() * COUT_FP; cout_aucun = yd.sum() * COUT_FN
print(f"part sollicitée optimale : {PART_SOLLICITEE:.1%} des clients")
print(f"coût / client : modèle {cout/len(yd):.3f} €   tout le monde {cout_tous/len(yd):.3f} €   personne {cout_aucun/len(yd):.3f} €")
parts = np.linspace(0.01, 1, 100); o = np.argsort(-oof); ys = yd[o]
couts = [((1-ys[:int(p*len(ys))]).sum()*COUT_FP + ys[int(p*len(ys)):].sum()*COUT_FN)/len(ys) for p in parts]
fig, ax = plt.subplots(figsize=(7, 3)); ax.plot(parts*100, couts, color="#3a6ea5")
ax.axvline(PART_SOLLICITEE*100, color="#c0392b", ls="--"); ax.set_xlabel("% des clients sollicités"); ax.set_ylabel("coût par client (€)")
plt.tight_layout(); plt.savefig(SORTIES / "cout_selon_part.png", dpi=120); plt.show()

### 2.7 Analyse d'erreurs *(interprète)*
Au seuil retenu, sur les scores hors-pli : **qui rate-t-on**, et à quoi ressemblent ces clients, statistiquement ?

In [ ]:
dec = np.zeros(len(yd), int); dec[np.argsort(-oof)[:n_sel]] = 1
vp, fp = int(((dec==1)&(yd==1)).sum()), int(((dec==1)&(yd==0)).sum())
fn, vn = int(((dec==0)&(yd==1)).sum()), int(((dec==0)&(yd==0)).sum())
print(f"vrais positifs {vp}   faux positifs {fp}   faux négatifs {fn}   vrais négatifs {vn}")
print(f"rappel {vp/(vp+fn):.1%}   précision {vp/(vp+fp):.1%}   coût des ratés : {fn*COUT_FN:.0f} €  coût des sollicitations inutiles : {fp*COUT_FP:.0f} €")

nb = S.BayesNaifQuantiles(N_BINS).fit(Xd, yd)          # pour lire les contributions (description, pas mesure)
contrib = nb.contributions(Xd)
groupes = {"vrais positifs": (dec==1)&(yd==1), "faux négatifs (ratés)": (dec==0)&(yd==1),
           "faux positifs": (dec==1)&(yd==0), "vrais négatifs": (dec==0)&(yd==0)}
desc = pd.DataFrame({g: {"clients": m.sum(),
                         "nb de variables qui « votent oui » (log-rapport > 0,1)": (contrib[m] > 0.1).sum(1).mean(),
                         "plus forte contribution": contrib[m].max(1).mean(),
                         "somme des contributions": contrib[m].sum(1).mean()} for g, m in groupes.items()}).T
desc.round(2)

In [ ]:
# Les ratés selon le décile de score : les positifs manqués sont-ils « presque trouvés » ou invisibles ?
dec_score = pd.qcut(pd.Series(oof).rank(method="first"), 10, labels=[f"D{i}" for i in range(1, 11)])
t = pd.DataFrame({"décile": dec_score, "y": yd}).groupby("décile", observed=True).y.agg(["count", "sum", "mean"])
t.columns = ["clients", "transactions", "taux"]; t["part des transactions"] = (t.transactions / t.transactions.sum()).round(3)
t.round(3)

**Lecture :** les transactions manquées ne sont pas des cas « presque trouvés » : elles ont **autant de
variables qui votent oui** qu'un non-client typique. Aucune variable ne crie, et le signal est diffus. Ce que
le modèle rate, il le rate faute d'information dans les 200 colonnes, pas faute de réglage.

## 3 · Mutations, robustesse et décision *(interprète)*

### 3.1 Trois mutations nommées — chacune en **une cellule**
| Mutation | Ce qu'elle change | Ce qu'on attend |
|---|---|---|
| **M1 — fuite par le découpage** | les tranches et la sélection sont apprises sur **toutes** les lignes, validation comprise | une AUC **trop belle** : la mutation qui casse le découpage |
| **M2 — cible mélangée** | la cible est permutée au hasard | AUC ≈ 0,5 : le protocole ne fabrique pas de signal |
| **M3 — moitié contre moitié** | apprendre sur une moitié, juger sur l'autre, puis l'inverse | les deux AUC proches : le modèle tient hors de ses données |

In [ ]:
# M1 — fuite par le découpage : le modèle voit la validation au moment d'apprendre ses tranches.
from sklearn.model_selection import StratifiedKFold
m_fuite = S.BayesNaifQuantiles(n_bins=256, lissage=0.5, voisinage=0).fit(Xd, yd)          # appris sur TOUT
aucs = [roc_auc_score(yd[va], m_fuite.decision_function(Xd[va])) for _, va in StratifiedKFold(N_PLIS, shuffle=True, random_state=GRAINE).split(Xd, yd)]
auc_ok, sd_ok, _ = S.score_cv(lambda: S.BayesNaifQuantiles(n_bins=256, lissage=0.5, voisinage=0), Xd, yd, N_PLIS)
print(f"M1 avec fuite : AUC {np.mean(aucs):.4f}   sans fuite (même modèle) : {auc_ok:.4f}   → écart fictif {np.mean(aucs)-auc_ok:+.4f}")
journal.noter("M1 mutation : fuite par le découpage (tranches apprises sur tout)", np.mean(aucs), np.std(aucs), note=f"faux gain {np.mean(aucs)-auc_ok:+.4f}", verdict="mutation")

In [ ]:
# M2 — cible mélangée : si le protocole est honnête, il ne doit rien trouver.
y_melange = np.random.default_rng(GRAINE).permutation(yd)
auc, sd, sec = S.score_cv(FABRIQUE, Xd, y_melange, N_PLIS)
print(f"M2 cible mélangée : AUC {auc:.4f} ± {sd:.4f}  (attendu ≈ 0,5)")
journal.noter("M2 mutation : cible mélangée", auc, sd, secondes=sec, verdict="mutation", note="contrôle du protocole")

In [ ]:
# M3 — moitié contre moitié.
rng = np.random.default_rng(GRAINE); a = rng.random(len(yd)) < 0.5
ab = roc_auc_score(yd[~a], S.score(FABRIQUE().fit(Xd[a], yd[a]), Xd[~a]))
ba = roc_auc_score(yd[a], S.score(FABRIQUE().fit(Xd[~a], yd[~a]), Xd[a]))
print(f"M3 A→B : {ab:.4f}   B→A : {ba:.4f}   écart {abs(ab-ba):.4f}")
journal.noter("M3 mutation : moitié contre moitié", (ab+ba)/2, abs(ab-ba)/2, verdict="mutation", note=f"A→B {ab:.4f} / B→A {ba:.4f}")

### 3.2 Test de robustesse
- **bruit** : on ajoute aux clients du test interne un bruit gaussien de 5 %, 10 %, 25 % de l'écart-type de
  chaque variable (erreurs de mesure, dérive) ;
- **variables perdues** : on retire les 10 variables les plus fortes (une source qui tombe) ;
- **moins de données** : on apprend sur 10 %, 25 %, 50 %, 100 % de la partie développement.

In [ ]:
modele = FABRIQUE().fit(Xd, yd)
ref_test = roc_auc_score(yt, S.score(modele, Xt))
rob = [("aucune perturbation", ref_test)]
rng = np.random.default_rng(GRAINE); sdv = Xd.std(0)
for f in [0.05, 0.10, 0.25]:
    rob.append((f"bruit {int(f*100)} % de l'écart-type", roc_auc_score(yt, S.score(modele, Xt + rng.normal(0, 1, Xt.shape).astype(np.float32) * sdv * f))))
top10 = [VARS.index(v) for v in force.index[:10]]; reste = [j for j in range(len(VARS)) if j not in top10]
m_sans = make_pipeline(S.SelectionAUC(min(K_VARIABLES, len(reste))), S.BayesNaifQuantiles(N_BINS)).fit(Xd[:, reste], yd)
rob.append(("sans les 10 variables les plus fortes", roc_auc_score(yt, S.score(m_sans, Xt[:, reste]))))
for part in [0.10, 0.25, 0.50]:
    sub = rng.random(len(yd)) < part
    rob.append((f"appris sur {int(part*100)} % des données", roc_auc_score(yt, S.score(FABRIQUE().fit(Xd[sub], yd[sub]), Xt))))
rob = pd.DataFrame(rob, columns=["situation", "AUC test interne"]).round(4)
rob.to_csv(SORTIES / "robustesse.csv", index=False); rob

### 3.3 Le test interne — touché une seule fois

In [ ]:
s_test = S.score(modele, Xt)
auc_test = roc_auc_score(yt, s_test)
n_t = int(round(PART_SOLLICITEE * len(yt))); d_t = np.zeros(len(yt), int); d_t[np.argsort(-s_test)[:n_t]] = 1
fp_t, fn_t = int(((d_t==1)&(yt==0)).sum()), int(((d_t==0)&(yt==1)).sum())
plancher_test = roc_auc_score(yt, regle.decision_function(Xt))
print(f"AUC test interne : modèle {auc_test:.4f}   plancher {plancher_test:.4f}   gain {auc_test-plancher_test:+.4f}")
print(f"au seuil ({PART_SOLLICITEE:.1%} sollicités) : rappel {((d_t==1)&(yt==1)).sum()/yt.sum():.1%}, "
      f"coût {(fp_t*COUT_FP+fn_t*COUT_FN)/len(yt):.3f} €/client contre {min((1-yt).sum()*COUT_FP, yt.sum()*COUT_FN)/len(yt):.3f} € sans modèle")
for l in journal.lignes:
    if l["essai"].startswith("E8"): l["auc_test_interne"] = round(auc_test, 4)
    if l["essai"].startswith("E0"): l["auc_test_interne"] = round(plancher_test, 4)

### 3.4 Décision finale sur les 40 000 clients
Le modèle est réappris sur **toutes** les lignes de `train.csv` (développement + test interne), puis on sollicite
la même part de clients que celle qui minimise le coût. Sortie : `ID_code,target` avec target = 1 « solliciter ».

In [ ]:
t = time.perf_counter()
final = FABRIQUE().fit(X, y)
s_dec = S.score(final, decider[VARS].to_numpy(np.float32))
n_dec = int(round(PART_SOLLICITEE * len(s_dec)))
decision = np.zeros(len(s_dec), int); decision[np.argsort(-s_dec)[:n_dec]] = 1
pd.DataFrame({S.ID: decider[S.ID], S.CIBLE: decision}).to_csv(SORTIES / "decision.csv", index=False)
pd.DataFrame({S.ID: decider[S.ID], "score": s_dec}).to_csv(SORTIES / "scores.csv", index=False)
print(f"{decision.sum()} clients sollicités sur {len(decision)} ({decision.mean():.1%}) — {time.perf_counter()-t:.1f} s")

In [ ]:
tab = journal.tableau(); tab["vs_plancher"] = (tab.auc_cv - PLANCHER).round(4)
tab.to_csv(SORTIES / "essais.csv", index=False)
json.dump({"fichier": str(TRAIN), "lignes": int(len(y)), "plancher_cv": PLANCHER, "modele": NOM_RETENU,
           "k": K_VARIABLES, "auc_cv": float(courbe[courbe.k == K_VARIABLES].auc.iloc[0]), "auc_test_interne": auc_test,
           "plancher_test_interne": plancher_test, "part_sollicitee": PART_SOLLICITEE,
           "cout_fp": COUT_FP, "cout_fn": COUT_FN, "duree_notebook_s": time.perf_counter() - t0},
          open(SORTIES / "resultats.json", "w"), indent=2, ensure_ascii=False)
print(f"durée totale : {time.perf_counter()-t0:.0f} s"); tab